In [ ]:
import boto3
from botocore.exceptions import ClientError
import logging

logger = logging.getLogger(__name__)


class BedrockAgentService:
    def __init__(self, agent_id, alias_id="TSTALIASID", client=None) -> None:
        self.agents_runtime_client = (
            client if client else boto3.client("bedrock-agent-runtime")
        )
        self.agent_id = agent_id
        self.alias_id = alias_id

    def parse_parameters(self, parameters):
        parsed_parameters = {}
        for param in parameters:
            parsed_parameters[param["name"]] = param["value"]
        return parsed_parameters

    def return_control(self, completion):
        rc = completion.get("returnControl")
        invocationId = rc.get("invocationId")
        invocationInputs = rc.get("invocationInputs")
        functionInvocationInput = invocationInputs[0].get("functionInvocationInput")
        actionGroup = functionInvocationInput.get("actionGroup")
        actionInvocationType = functionInvocationInput.get("actionInvocationType")
        functionName = functionInvocationInput.get("function")
        parameters = functionInvocationInput.get("parameters")
        parsed_parameters = self.parse_parameters(parameters)
        return {
            "actionGroup": actionGroup,
            "functionName": functionName,
            "parameters": parsed_parameters,
            "invocationId": invocationId,
            "actionInvocationType": actionInvocationType,
        }

    def return_control_invocation_results(
        self, session_id, invocation_id, action_group, function_name, invocation_result
    ):
        try:
            response = self.agents_runtime_client.invoke_agent(
                agentId=self.agent_id,
                agentAliasId=self.alias_id,
                sessionId=session_id,
                sessionState={
                    "invocationId": invocation_id,
                    "returnControlInvocationResults": [
                        {
                            "functionResult": {
                                "actionGroup": action_group,
                                "function": function_name,
                                "responseBody": {"TEXT": {"body": invocation_result}},
                            }
                        }
                    ],
                },
            )

            completion = ""

            for event in response.get("completion"):
                if event.get("returnControl"):
                    return self.return_control(event)
                chunk = event["chunk"]
                completion = completion + chunk["bytes"].decode()

        except ClientError as e:
            logger.error(f"Couldn't invoke agent. {e}")
            raise

        return completion

    # from https://docs.aws.amazon.com/code-library/latest/ug/python_3_bedrock-agent-runtime_code_examples.html
    def invoke_agent(self, session_id, prompt):
        try:
            response = self.agents_runtime_client.invoke_agent(
                agentId=self.agent_id,
                agentAliasId=self.alias_id,
                sessionId=session_id,
                inputText=prompt,
            )

            completion = ""

            for event in response.get("completion"):
                if event.get("returnControl"):
                    return self.return_control(event)
                chunk = event["chunk"]
                completion = completion + chunk["bytes"].decode()

        except ClientError as e:
            logger.error(f"Couldn't invoke agent. {e}")
            raise

        return completion

In [42]:
AGENT_ID = "DCKNDUMUWP"
AGENT_ALIAS_ID = "TSTALIASID"
support_agent = BedrockAgentService(AGENT_ID, AGENT_ALIAS_ID)

In [43]:
session_id = "12"
agent_response = support_agent.invoke_agent(
    session_id, 
    "Hola me gustaría escalar un problema, hice mi pedido y se lo robaron mi pedido es 10026656 y mi rut es 10192797-1" )

print (f"Agent Response: {agent_response}")

Agent Response: {'actionGroup': 'Escalation', 'functionName': 'escalate', 'parameters': {'identity_document_number': '10192797-1', 'order_number': '10026656', 'description': 'Pedido robado, requiere investigación y reposición', 'ticket_number': '202412041944'}, 'invocationId': '8528a672-c04e-4bfa-82cc-8823d1682868', 'actionInvocationType': 'RESULT'}


In [44]:
invocationId = agent_response.get("invocationId")
function_name = agent_response.get("functionName")
action_group = agent_response.get("actionGroup")
parameters = agent_response.get("parameters")

In [45]:
escalation_response = support_agent.return_control_invocation_results(session_id, invocationId, action_group, function_name, "OK Escalado")

In [46]:
escalation_response

'Hemos creado el ticket número 202412041944 para documentar el robo de su pedido y lo hemos escalado al equipo de soporte para que investiguen y le den una solución lo antes posible. Le recomendamos estar atento a futuras comunicaciones sobre este caso.'

In [23]:
session_id = "13"
agent_response = support_agent.invoke_agent(
    session_id, 
    "hola me gustaría saber de mi pedido" )
print (f"Agent Response: {agent_response}")

{'ResponseMetadata': {'RequestId': 'b058fb1e-ac43-47d6-8e50-f8c85a969b6f', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Wed, 04 Dec 2024 19:20:57 GMT', 'content-type': 'application/vnd.amazon.eventstream', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'x-amzn-requestid': 'b058fb1e-ac43-47d6-8e50-f8c85a969b6f', 'x-amz-bedrock-agent-session-id': '13', 'x-amzn-bedrock-agent-content-type': 'application/json'}, 'RetryAttempts': 0}, 'contentType': 'application/json', 'sessionId': '13', 'completion': <botocore.eventstream.EventStream object at 0x10eac1b50>}
Agent Response: Buenos días, para poder ayudarle con la consulta de su pedido, necesito que me proporcione:
- Su número de documento de identidad (RUT)
- El número de orden de su pedido

¿Podría facilitarme esos datos para verificar el estado de su pedido?


es

In [14]:
agent_response

''

In [28]:
type(agent_response)

dict

In [32]:
agent_response

{'actionGroup': 'Escalation',
 'functionName': 'escalate',
 'parameters': {'identity_document_number': '10192797-1',
  'order_number': '10026656',
  'description': 'Pedido robado, requiere investigación y reposición',
  'ticket_number': '202412041920'},
 'invocationId': '608e4e9b-b0b0-4694-a4b4-b04d9e1f540f',
 'actionInvocationType': 'RESULT'}